## LSTM-AE (LSTM AutoEncoder) で分析してみる

In [2]:
pip install torch

Note: you may need to restart the kernel to use updated packages.


In [3]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.11.0
False


In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import classification_report

In [5]:
df = pd.read_csv('../data/raw/sensor.csv')
df.head()

,Unnamed: 0,timestamp,sensor_00,sensor_01,sensor_02,sensor_03,sensor_04,sensor_05,sensor_06,sensor_07,...,sensor_43,sensor_44,sensor_45,sensor_46,sensor_47,sensor_48,sensor_49,sensor_50,sensor_51,machine_status
0,0,2018-04-01 00:00:00,2.465394,47.09201,53.2118,46.310760,634.3750,76.45975,13.41146,16.13136,...,41.92708,39.641200,65.68287,50.92593,38.194440,157.9861,67.70834,243.0556,201.3889,NORMAL
1,1,2018-04-01 00:01:00,2.465394,47.09201,53.2118,46.310760,634.3750,76.45975,13.41146,16.13136,...,41.92708,39.641200,65.68287,50.92593,38.194440,157.9861,67.70834,243.0556,201.3889,NORMAL
2,2,2018-04-01 00:02:00,2.444734,47.35243,53.2118,46.397570,638.8889,73.54598,13.32465,16.03733,...,41.66666,39.351852,65.39352,51.21528,38.194443,155.9606,67.12963,241.3194,203.7037,NORMAL
3,3,2018-04-01 00:03:00,2.460474,47.09201,53.1684,46.397568,628.1250,76.98898,13.31742,16.24711,...,40.88541,39.062500,64.81481,51.21528,38.194440,155.9606,66.84028,240.4514,203.1250,NORMAL
4,4,2018-04-01 00:04:00,2.445718,47.13541,53.2118,46.397568,636.4583,76.58897,13.35359,16.21094,...,41.40625,38.773150,65.10416,51.79398,38.773150,158.2755,66.55093,242.1875,201.3889,NORMAL


In [6]:
df['machine_status'].value_counts()

machine_status
NORMAL        205836
RECOVERING     14477
BROKEN             7
Name: count, dtype: int64

欠損値の補完

In [7]:
sensors = ['sensor_04', 'sensor_39', 'sensor_41', 'sensor_42']
df[sensors] = df[sensors].interpolate()

windowを作る
<br>学習用データとして、NORMAL期間のみのwindowが入ったリストを作らなければならない。

In [8]:
def create_windows(data, window_size, stride=1):
    windows=[]
    for i in range(0, len(data) - window_size + 1, stride):
        window = data[i:i+window_size]
        windows.append(window)
    return np.array(windows)

In [9]:
normal_df = df[df['machine_status'] == 'NORMAL']
gaps = normal_df.index.to_series().diff() != 1
group_idx = gaps.cumsum()

In [10]:
normal_df['group_idx'] = group_idx
all_normal_windows = []
for key, segment in normal_df.groupby('group_idx'):
    data = segment[sensors].to_numpy()
    windows = create_windows(data, 60, stride = 1)
    all_normal_windows.append(windows)

In [11]:
all_normal_windows = np.concatenate(all_normal_windows, axis = 0)
all_normal_windows.shape

(205364, 60, 4)

正規化を行う
<br>sensor_04 は 600 前後、sensor_39 は 30 前後と、センサーごとにスケールが全然違うため

In [12]:
from sklearn.preprocessing import StandardScaler

X_2d = all_normal_windows.reshape(-1,4)

scaler = StandardScaler()
X_2d_scaled = scaler.fit_transform(X_2d)

X_scaled = X_2d_scaled.reshape(-1, 60, 4)
X_scaled.shape


(205364, 60, 4)

Tensor への変換と DataLoader の作成 

1バッチ64時刻分
<br>200000ぐらいある時系列を64個ずつ切り分けていく。
<br>なぜ切り分けるかというと、64時刻ずつfor文で一個一個学習する場合、一回一回のデータ量がRAM(メモリ)に乗るから、はやく処理できる。もし、切り分けずに220000時刻全てを一気に学習しようと思うと、一回そのデータ全てがRAMに乗ろうとして、でも乗り切らないからパソコンが勝手にディスク上にそのデータを置こうとする。ディスクは処理がとても遅いから、学習に膨大な時間がかかってしまうことになる・

In [13]:
# NumPy array → PyTorch Tensor
X_tensor = torch.tensor(X_scaled, dtype = torch.float32)

# TensorDataset → DataLoader
dataset = TensorDataset(X_tensor, X_tensor)
loader = DataLoader(dataset, batch_size = 64, shuffle = True)

In [14]:
class Encoder(nn.Module):
    def __init__(self, n_features, hidden_size):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden_size, batch_first=True)

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        return h_n.squeeze(0)   # (1, batch, hidden) → (batch, hidden)


class Decoder(nn.Module):
    def __init__(self, hidden_size, n_features, seq_len):
        super().__init__()
        self.seq_len = seq_len
        self.lstm = nn.LSTM(hidden_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, n_features)

    def forward(self, z):
        z = z.unsqueeze(1).repeat(1, self.seq_len, 1)  # (batch, 60, hidden)
        out, _ = self.lstm(z)
        return self.fc(out)     # (batch, 60, 4)


class LSTMAE(nn.Module):
    def __init__(self, n_features, hidden_size, seq_len):
        super().__init__()
        self.encoder = Encoder(n_features, hidden_size)
        self.decoder = Decoder(hidden_size, n_features, seq_len)

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)


In [15]:
model = LSTMAE(n_features=4, hidden_size=64, seq_len=60)

In [16]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [17]:
n_epochs = 20

for epoch in range(n_epochs):
    model.train()
    total_loss = 0
    for X_batch, _ in loader:
        output = model(X_batch)
        loss = criterion(output, X_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(loader)
    print(f"Epoch {epoch+1}/{n_epochs}  loss: {avg_loss:.4f}")

Epoch 1/20  loss: 0.3840
Epoch 2/20  loss: 0.2653
Epoch 3/20  loss: 0.2434
Epoch 4/20  loss: 0.2004
Epoch 5/20  loss: 0.1680
Epoch 6/20  loss: 0.1400
Epoch 7/20  loss: 0.1236
Epoch 8/20  loss: 0.1203
Epoch 9/20  loss: 0.1021
Epoch 10/20  loss: 0.0922
Epoch 11/20  loss: 0.0981
Epoch 12/20  loss: 0.0854
Epoch 13/20  loss: 0.0753
Epoch 14/20  loss: 0.0744
Epoch 15/20  loss: 0.0649
Epoch 16/20  loss: 0.0611
Epoch 17/20  loss: 0.0603
Epoch 18/20  loss: 0.0551
Epoch 19/20  loss: 0.0567
Epoch 20/20  loss: 0.0494


全体データで評価

In [18]:
all_windows = create_windows(df[sensors], 60, stride=1)
all_windows_2D = all_windows.reshape(-1,4)
all_windows_2D_scaled = scaler.transform(all_windows_2D)
all_windows_scaled = all_windows_2D_scaled.reshape(-1, 60, 4)

In [19]:
X_all_tensor = torch.tensor(all_windows_scaled, dtype = torch.float32)

In [20]:
all_dataset = TensorDataset(X_all_tensor)
all_loader = DataLoader(all_dataset, batch_size = 256 )

同様に評価時もバッチに切り分ける

In [21]:
all_errors = []
model.eval()
with torch.no_grad():
    for (X_batch,) in all_loader:
        output = model(X_batch)
        batch_errors = (output - X_batch) ** 2
        batch_errors = batch_errors.mean(dim = (1,2))
        all_errors.append(batch_errors)

errors = np.concatenate(all_errors)
print(errors.shape)

(220261,)


ここでerrorsには各窓に対して、一つの誤差スカラーが存在するというseriesみたいな形状になってる

In [22]:
labels = df['machine_status'].values
window_labels = labels[59:]
#これで完全にerrorsとwindow_labelsのインデックスが対応する

normal_errors = errors[window_labels == 'NORMAL']
threshold = normal_errors.mean() + 1 * normal_errors.std()

y_pred = (errors > threshold).astype(int)
y_true = (window_labels != 'NORMAL').astype(int)
print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.94      1.00      0.97    205777
           1       0.66      0.08      0.14     14484

    accuracy                           0.94    220261
   macro avg       0.80      0.54      0.55    220261
weighted avg       0.92      0.94      0.91    220261



In [23]:
print("NORMAL    誤差:", errors[window_labels == 'NORMAL'].mean().round(4))
print("RECOVERING誤差:", errors[window_labels == 'RECOVERING'].mean().round(4))

NORMAL    誤差: 0.0979
RECOVERING誤差: 4.7293


確かにrecoveringの誤差はnormalよりも大きい、それなのに、
<br>閾値 = 平均 + k * 標準偏差という設計では評価の精度が上がらない。これはnormalとして捉えられている窓（そのうちの最後の時系列のラベルがnormalだから）の中にその中身のほとんどがrecoveringで構成されていて、誤差がとても大きい窓が存在していて、そもそもnormal_errors.std()がそれに引っ張られてとても大きくなってしまっている可能性が考えられる。
<br>また、normal_errors.mean()についても、そのようなほぼrecoveringで構成されたnormal窓の値によって引っ張られて、大きな値となっている可能性がある（たとえばそれが中央値どころか95
%点よりも大きい値だったりしたら、いくらstdに掛け合わせる係数を小さくしても閾値はmeanにあるから、閾値はそれ以上下がらない）

In [26]:
normal_errors = errors[window_labels == 'NORMAL']
threshold = np.percentile(normal_errors, 95)  # NORMALの95%点


y_pred = (errors > threshold).astype(int)
y_true = (window_labels != 'NORMAL').astype(int)
print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.99      0.95      0.97    205777
           1       0.53      0.81      0.65     14484

    accuracy                           0.94    220261
   macro avg       0.76      0.88      0.81    220261
weighted avg       0.96      0.94      0.95    220261



閾値にnormal_errorsの95%点を用いるとrecallの値が大幅に改善された。

## まとめ

| 手法 | Precision | Recall | F1 |
|---|---|---|---|
| ARIMA 4センサーOR | 0.85 | 0.94 | 0.89 |
| Isolation Forest（contamination=0.015） | 0.82 | 0.98 | 0.89 |
| ARIMA + IF アンサンブル | 0.78 | 1.00 | 0.87 |
| LSTM-AE（今回） | 0.53 | 0.81 | 0.65 |


- ARIMAのresidualsについて、4センサーそれぞれ目視して、閾値を決定してそれをORで統合する方法が今回は良い結果となった。
- Isolation Forestも良さげ。
- LSTM-AEについて、今回は学習回数が少なく（epoch=20で止めたが、まだlossは下降中だった）、アーキテクチャが最小構成（hidden_size(隠れ状態のベクトルの長さ) = 64, 単層LSTM）だったためまだ改善の余地は残されている。
